# Proyecto Final — Asistente de regalos
## Calidad de datos
Entendimiento y limpieza del dataset Online Retail II.

### 1. Carga de datos

In [42]:
import pandas as pd

hojas = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name=None)  # el Excel trae 2 hojas, una por período
df = pd.concat(hojas.values(), ignore_index=True)  # las uno en una sola tabla
print(df.shape)

(1067371, 8)


**Hallazgo:** el dataset cargó con 1.067.371 filas y 8 columnas,
coincide con lo documentado oficialmente para Online Retail II.

**Diccionario de variables** (para no marearse con los nombres de columna más abajo):

| Columna | Qué es | Notas |
|---|---|---|
| invoice_no | Número de factura/pedido | Si empieza con "C" es cancelación |
| stock_code | Código del producto | Algunos códigos no son productos reales (POST, DOT, M, BANK CHARGES, AMAZONFEE, B) |
| description | Descripción del producto | Texto libre, puede venir vacío |
| quantity | Cantidad comprada en esa línea | Puede ser negativa (cancelaciones o ajustes) |
| invoice_date | Fecha y hora de la transacción | Rango 2009-2011 |
| unit_price | Precio de una unidad | En libras esterlinas (GBP); puede venir en 0 |
| customer_id | Identificador del cliente | Puede venir vacío |
| country | País del cliente | Mayoría Reino Unido |

### 2. Exploración inicial
¿Cuántos datos faltan, qué tipos de dato hay, qué rango de fechas cubre?

In [43]:
df.info()  # tipos de dato de cada columna
display(df.isna().sum())  # cuántos valores faltan por columna

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 78.8+ MB


Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

**Hallazgo:** las columnas vinieron como `Invoice`, `Price` y
`Customer ID` (no `InvoiceNo`, `UnitPrice`, `CustomerID`). Hay
243.007 filas sin `Customer ID` (22,8%) y 4.382 sin `Description`
(0,4%) — ambos se van a tratar en la limpieza.

### 3. Normalización de columnas
Las columnas se renombran para trabajar con un estándar fijo (snake_case).

In [44]:
df = df.rename(columns={
    'Invoice': 'invoice_no', 'StockCode': 'stock_code', 'Description': 'description',
    'Quantity': 'quantity', 'InvoiceDate': 'invoice_date', 'Price': 'unit_price',
    'Customer ID': 'customer_id', 'Country': 'country'
})
df.columns.tolist()

['invoice_no',
 'stock_code',
 'description',
 'quantity',
 'invoice_date',
 'unit_price',
 'customer_id',
 'country']

**Hallazgo:** las 8 columnas quedaron normalizadas en snake_case.
A partir de acá, todo el código usa estos nombres.

### 4. Mirando la tabla y sus rangos

In [45]:
display(df.head())  # miro filas reales antes de cualquier estadística

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


**Hallazgo:** la estructura es 1 fila por producto dentro de una
factura (mismo invoice_no se repite). La primera factura (489434)
parecía, mirando solo estas 5 filas, tener todas sus cantidades
en múltiplos de 6 — hay que verificar si head() la cortó.

In [84]:
display(df[df['invoice_no'].astype(str) == '489434'])

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom


**Hallazgo corregido:** la factura 489434 tiene 8 líneas (no 5, como
parecía con head()). De esas, 7 son múltiplo de 6 y 1 no (cantidad 10)
— o sea, 87,5% del patrón bulk, no el 100% que se había dicho antes.
También confirma que códigos con letra al final (79323P, 79323W)
pueden ser productos reales (variantes de color), no solo ajustes
administrativos.

In [85]:
print(df['invoice_no'].apply(type).value_counts())

invoice_no
<class 'int'>    1047871
<class 'str'>      19500
Name: count, dtype: int64


**Hallazgo adicional:** invoice_no también mezcla tipos de dato en 
el Excel crudo — 1.047.871 filas como entero, 19.500 como texto 
(las que empiezan con "C", que la letra obliga a ser string). Mismo 
patrón que stock_code. Confirma que normalizar_tipos() es necesario 
para TODAS las columnas de identificador, no solo stock_code.

### 5. Rangos y valores extremos

In [47]:
display(df.describe())  # count, media, min, max, cuartiles de las columnas numéricas

,quantity,invoice_date,unit_price,customer_id
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394029,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


**Hallazgo:** quantity tiene un rango de -80.995 a +80.995, mientras
que unit_price llega a -53.594,36. Son valores extremos poco
habituales — hay que investigarlos antes de decidir cómo tratarlos.

### 6. Casos sospechosos: cantidad o precio no válidos

In [48]:
display(df[(df['quantity'] <= 0) | (df['unit_price'] <= 0)].head(10))  # miro una muestra
print('cantidad <= 0:', (df['quantity'] <= 0).sum())
print('unit_price <= 0:', (df['unit_price'] <= 0).sum())

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom


cantidad <= 0: 22950
unit_price <= 0: 6207


**Hallazgo:** 22.950 filas tienen quantity <= 0, más que las 19.494
cancelaciones formales (invoice_no con "C") — hay ~3.456 filas con
cantidad negativa o cero que no están marcadas como cancelación
oficial (se investiga en la sección 9). El filtro quantity > 0 las
saca a todas de todos modos. Aparte, 6.207 filas tienen
unit_price <= 0 (incluye los 4 casos de "Adjust bad debt" de la
sección 7, más ceros).

### 6.1 Duplicados exactos

In [49]:
print(df.duplicated().sum(), "filas duplicadas exactas")  # cuenta filas idénticas en TODAS sus columnas

34335 filas duplicadas exactas


In [50]:
display(df[df.duplicated(keep=False)].sort_values(['invoice_no', 'stock_code']).head(10))  # ejemplo real, no solo el número

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


**Hallazgo:** 34.335 filas duplicadas exactas (3,2% del dataset).
Un ejemplo real: "VINTAGE SNAKES & LADDERS" aparece 3 veces en la
misma factura, cada una con quantity=1 — no se ve como 1 compra
de 3 unidades en una sola línea, sino como 3 líneas separadas.

### 6.2 ¿Se repite el patrón en los 34.335?

In [51]:
duplicados = df[df.duplicated(keep=False)]
print(duplicados['quantity'].value_counts().head())  # ¿la mayoría tiene quantity=1, como en el ejemplo?

repeticiones = duplicados.groupby(['invoice_no', 'stock_code']).size()  # cuántas veces se repite cada combinación factura+producto
print(repeticiones.describe())

quantity
1     31263
2      9661
12     4896
6      4300
3      3776
Name: count, dtype: int64
count    32597.000000
mean         2.062828
std          0.377345
min          2.000000
25%          2.000000
50%          2.000000
75%          2.000000
max         20.000000
dtype: float64


**Hallazgo:** el 75% de las combinaciones factura+producto
duplicadas se repite exactamente 2 veces (mediana = 2). Si
descartáramos duplicados quedándonos con 1 sola fila, perderíamos
la cantidad real que se llevó el cliente (ej. quedaría quantity=1
en vez de 3).

**Decisión:** no se usa drop_duplicates() a secas. Se agrupan las
filas duplicadas y se suma quantity, para conservar la cantidad
real que se llevó el cliente en vez de perderla al quedarse con
una sola fila.

### 7. Investigando el precio negativo extremo

In [52]:
display(df[df['unit_price'] < 0][['stock_code', 'description', 'unit_price']].drop_duplicates())  # ¿qué productos tienen precio negativo?

,stock_code,description,unit_price
179403,B,Adjust bad debt,-53594.36
276274,B,Adjust bad debt,-44031.79
403472,B,Adjust bad debt,-38925.87
825444,B,Adjust bad debt,-11062.06


**Hallazgo:** stock_code "B" = "Adjust bad debt" — es un ajuste
contable por deuda incobrable, no una venta real. Se suma a la
lista de códigos no-producto a sacar en la limpieza (junto con
POST, DOT, M, BANK CHARGES, AMAZONFEE).

### 7.1 Auditoría de códigos que no siguen el patrón estándar

In [53]:
codigos_raros = df[~df['stock_code'].astype(str).str.match(r'^\d{5}$')]['stock_code'].unique()  # códigos que NO son 5 dígitos seguidos (astype(str): stock_code mezcla int y str en el Excel crudo)
print(len(codigos_raros), "códigos distintos no son 5 dígitos seguidos")
print(sorted(codigos_raros)[:30])  # primeros 30, para no inundar la pantalla

1715 códigos distintos no son 5 dígitos seguidos
['10002R', '10123C', '10123G', '10124A', '10124C', '10124G', '15044A', '15044B', '15044C', '15044D', '15056BL', '15056N', '15056P', '15056bl', '15056n', '15056p', '15058A', '15058B', '15058C', '15059A', '15060A', '15060B', '15060b', '16020C', '16065A', '16151A', '16155H', '16156A', '16156L', '16156N']


**Hallazgo:** 1.715 códigos no siguen el patrón de 5 dígitos. La
mayoría son variantes de color/tamaño de un producto base (ej.
10124A, 10124C, 10124G) — productos reales, no administrativos.

In [54]:
tipos = df['stock_code'].apply(type).value_counts()  # confirmo por qué hizo falta el astype(str) de arriba
print(tipos)

stock_code
<class 'int'>    932385
<class 'str'>    134986
Name: count, dtype: int64


**Hallazgo:** stock_code en el Excel crudo mezcla tipos de dato —
932.385 filas como número entero, 134.986 como texto. Excel guarda
así las celdas puramente numéricas (ej. 20725) distinto de las
alfanuméricas (ej. 85123A). Esto confirma por qué normalizar_tipos()
fuerza todo a texto antes de cualquier otra operación — no es solo
prolijidad, es necesario para que el código no se rompa.

### 7.2 ¿Son duplicados por mayúscula/minúscula?

In [55]:
print(df['stock_code'].nunique(), "códigos distintos (mayúscula y minúscula por separado)")  # cuenta cada variante de caso como distinta
print(df['stock_code'].astype(str).str.upper().nunique(), "códigos distintos después de unificar mayúsculas")  # unifico y cuento de nuevo

5305 códigos distintos (mayúscula y minúscula por separado)
5132 códigos distintos después de unificar mayúsculas


**Hallazgo:** 5.305 códigos distintos bajan a 5.132 al unificar
mayúsculas — 173 eran el mismo producto escrito distinto (ej.
15056BL y 15056bl). Confirma que normalizar con .str.upper() en
la limpieza es necesario, no una precaución de más.

### 7.3 Acotar los candidatos reales a revisar

In [56]:
precio_cero_siempre = df.groupby('stock_code')['unit_price'].apply(lambda s: (s == 0).all())  # True si TODAS las ventas de ese código tienen precio 0
sospechosos = precio_cero_siempre[precio_cero_siempre].index.tolist()  # me quedo solo con los códigos marcados True
print(len(sospechosos), "códigos con TODAS sus filas en precio 0 (candidatos reales a revisar)")

373 códigos con TODAS sus filas en precio 0 (candidatos reales a revisar)


**Hallazgo:** 373 códigos tienen precio 0 en el 100% de sus ventas.
La mayoría son productos reales de 5 dígitos (muestras o
promociones) — el filtro unit_price > 0 ya los saca a todos sin
necesitar código nuevo.

### 7.4 Revisar los más raros entre los sospechosos

In [57]:
sospechosos_especiales = ['C3', 'GIFT', 'gift_0001_60', 'gift_0001_90', 'DCGS0006', 'DCGS0016', 'DCGS0027',
                           'DCGS0036', 'DCGS0039', 'DCGS0053', 'DCGS0055', 'DCGS0056', 'DCGS0057', 'DCGS0059',
                           'DCGS0060', 'DCGS0066P', 'DCGS0067', 'DCGS0071', 'DCGS0073', 'DCGS0074',
                           'DCGSLBOY', 'DCGSLGIRL']
display(df[df['stock_code'].isin(sospechosos_especiales)][['stock_code', 'description']].drop_duplicates())  # miro sus descripciones reales

,stock_code,description
75138,gift_0001_60,NaN
75141,gift_0001_90,NaN
80337,DCGS0006,NaN
80338,DCGS0016,NaN
80339,DCGS0027,NaN
80342,DCGS0036,NaN
80361,DCGS0039,NaN
80953,DCGS0060,NaN
81143,DCGS0056,NaN
82846,DCGS0059,NaN


**Hallazgo:** de los 373 códigos con precio siempre en 0, 22 (los
"gift_", "GIFT", "DCGS0XXX", "C3") tienen descripción vacía o solo
"ebay" — sin evidencia de ser productos identificables.

**Decisión:** no se agregan a la lista de códigos no-producto.
catalogo_productos() ya descarta cualquier stock_code sin
descripción, así que estos 22 nunca llegan a mostrarse ni a
recomendarse en el asistente.

### 8. País y cancelaciones

In [58]:
display(df['country'].value_counts().head(10))  # top 10 países
print(df['invoice_no'].astype(str).str.startswith('C').sum(), "filas de cancelación")  # cuántas son devoluciones

country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64

19494 filas de cancelación


**Hallazgo:** United Kingdom es el 91,9% de las filas (981.330 de
1.067.371). Hay 19.494 filas de cancelación (1,8%).

### 9. Cantidad negativa SIN ser cancelación formal

In [59]:
mask = (df['quantity'] < 0) & (~df['invoice_no'].astype(str).str.startswith('C'))  # cantidad negativa Y NO es cancelación formal
print(mask.sum(), "filas con cantidad negativa que no son cancelación 'C'")
display(df[mask].head(10))  # miro una muestra para tener una primera idea

3457 filas con cantidad negativa que no son cancelación 'C'


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom


In [60]:
print('todas con unit_price == 0:', (df[mask]['unit_price'] == 0).all())  # ¿se cumple en TODAS, o solo en la muestra que vi?
print('todas con customer_id nulo:', df[mask]['customer_id'].isna().all())

todas con unit_price == 0: True
todas con customer_id nulo: True


**Hallazgo:** las 3.457 filas con cantidad negativa que no son
cancelación formal tienen el 100% unit_price = 0 y 100%
customer_id nulo — confirmado sobre las 3.457 filas completas, no
solo sobre una muestra. Son ajustes de inventario escritos a
mano ("lost", "damages", "short", "invcd as..."), no ventas ni
cancelaciones de cliente. El filtro unit_price > 0 (ya planeado)
las saca a todas — no hace falta un filtro nuevo para esto.

### 9.1 Reconciliando la diferencia de 1 fila (3.456 vs 3.457)

In [61]:
mask_rara = df['invoice_no'].astype(str).str.startswith('C') & (df['quantity'] >= 0)  # facturas "C" con cantidad NO negativa
print(mask_rara.sum(), "facturas 'C' con cantidad >= 0")
display(df[mask_rara])  # veo de qué se trata

1 facturas 'C' con cantidad >= 0


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom


**Hallazgo:** la diferencia de 1 fila (3.457 vs 3.456) era una
factura "C" con stock_code "M" (Manual) y cantidad positiva — un
ajuste administrativo cargado con prefijo de cancelación, no una
cancelación real de cliente.

**Decisión:** no hace falta ningún cambio en el código de limpieza
— esta fila ya se excluye sola con el filtro de códigos no-producto
existente (stock_code "M" está en CODIGOS_NO_PRODUCTO).

### 10. ¿El patrón de la primera factura se repite en todo el dataset?

In [62]:
proporcion_multiplo = (df['quantity'] % 6 == 0).groupby(df['invoice_no']).mean()  # % de líneas múltiplo de 6, por factura
display(proporcion_multiplo.describe())

n_productos = df.groupby('invoice_no')['stock_code'].nunique()  # productos distintos por factura
es_bulk = (proporcion_multiplo >= 0.8) & (n_productos >= 5)  # 80%+ en múltiplos de 6 Y al menos 5 productos
print(f"% facturas bulk (proporción 80% + mínimo 5 productos): {es_bulk.mean():.1%}")

count    53628.000000
mean         0.352582
std          0.353950
min          0.000000
25%          0.000000
50%          0.250000
75%          0.647059
max          1.000000
Name: quantity, dtype: float64

% facturas bulk (proporción 80% + mínimo 5 productos): 7.1%


**Hallazgo:** la proporción de líneas en múltiplos de 6 por factura
tiene mediana 0,25 y percentil 75 de 0,647 — la mayoría de las
facturas no se acerca al patrón bulk. Con el criterio corregido
(80%+ de las líneas en múltiplos de 6, Y al menos 5 productos
distintos), el 7,1% de las facturas califica como bulk. Una sola
excepción en una factura de 1.110 productos no debería descartarla
— por eso se usa proporción, no el 100% exacto que se había
considerado al principio.

### 11. Impacto de conservar filas sin customer_id

In [63]:
con_cliente = df.dropna(subset=['customer_id'])  # como quedaría si exigimos cliente

print(f"Facturas totales: {df['invoice_no'].nunique()} | con cliente: {con_cliente['invoice_no'].nunique()}")
print(f"Productos distintos totales: {df['stock_code'].nunique()} | con cliente: {con_cliente['stock_code'].nunique()}")

Facturas totales: 53628 | con cliente: 44876
Productos distintos totales: 5305 | con cliente: 4646


**Hallazgo:** exigir customer_id elimina 8.752 facturas completas
(16,3%) y 659 productos distintos (12,4% del catálogo) que solo
aparecen en compras sin cliente registrado.

**Decisión:** se arman 2 datasets a partir de acá. Para RFM y
segmentación mayorista/minorista: pipeline_completo(conservar_sin_cliente=False)
— ahí SÍ hace falta saber quién compró. Para el modelo de
recomendación (matriz factura-producto):
pipeline_completo(conservar_sin_cliente=True) — el modelo no
pregunta quién compró, y descartar esas filas le esconde 1 de
cada 8 productos del catálogo.

## Conclusión: qué aprendimos antes de limpiar

El dataset tiene 1.067.371 filas y 8 columnas, consistente con lo
documentado oficialmente. Las columnas no venían estandarizadas,
así que las normalizamos a snake_case.

**Valores extremos investigados — 2 hipótesis razonables, no
confirmadas con certeza:**
1. Stock_code "B" trae la descripción "Adjust bad debt". Nuestra
   interpretación: es un ajuste contable por deuda incobrable, no
   una venta — apoyada en que el precio es negativo y enorme, pero
   no hay forma de confirmarlo con quien armó el dataset.
2. 3.457 filas con cantidad negativa, sin ser cancelación formal,
   traen descripciones como "lost", "damages", "short", con 100%
   precio=0 y 100% sin cliente. Nuestra lectura: pérdida o daño de
   inventario, no una compra — razonable dado el patrón, pero
   también una interpretación, no un hecho verificado.

Reino Unido concentra el 91,9% de las filas (hecho medido);
cancelaciones formales son el 1,8% (hecho medido).

**Códigos de producto:** 1.715 no siguen el patrón de 5 dígitos
(hecho medido). Revisando sus descripciones, la mayoría parecen
variantes de color o tamaño de un mismo producto (confirmado
viendo el texto real, ej. 79323P/79323W = "PINK/WHITE CHERRY
LIGHTS"). Unificar mayúsculas bajó los códigos distintos de 5.305
a 5.132 (hecho medido). De 373 códigos con precio siempre en 0,
22 no tienen descripción real — ya quedan afuera del catálogo
automáticamente.

**Duplicados exactos:** 34.335 filas (3,2%, hecho medido). Se
agrupa y suma `quantity` en vez de descartar, para no perder
cantidad real comprada.

**Patrón de "facturas bulk":** le llamamos así a una hipótesis de
reposición al por mayor, inferida de que las cantidades vienen en
múltiplos de 6 y hay varios productos distintos — no es un dato
confirmado, es un patrón que medimos (7,1% de las facturas, con
80%+ de líneas en múltiplos de 6 y mínimo 5 productos).

**Clientes sin identificar:** exigir customer_id perdería 8.752
facturas (16,3%) y 659 productos (12,4% del catálogo) — esto sí es
un hecho medido directo. Por eso se generan 2 datasets: uno exige
cliente (RFM), otro lo conserva (modelo de recomendación).

Con esto, la limpieza queda justificada con evidencia propia —
distinguiendo qué se midió directamente y qué es una interpretación
razonable de esa evidencia, no un hecho confirmado al 100%.

### 12. Aplicando la limpieza real
Corremos el pipeline de limpieza (src/data/clean.py) para generar los datasets finales.

In [64]:
import sys
sys.path.append('..')

from src.data import clean
import importlib
importlib.reload(clean)  # por si el módulo ya estaba cargado en memoria con una versión vieja
from src.data.clean import pipeline_completo

**Hallazgo:** la primera corrida del pipeline mostró 3 pasos
consecutivos con el mismo número de filas ("sin duplicados", "sin
descripción vacía", "con customer_id") — señal de que groupby()
estaba descartando de más. Por default, pandas tira las filas con
NaN en las columnas de agrupamiento, y en ese punto del pipeline
customer_id y description todavía tenían NaN.

**Decisión:** se agregó dropna=False al groupby() de limpiar_datos()
en clean.py, para que agrupe los NaN entre sí en vez de descartarlos.
Con el fix, cada filtro saca solo lo que le corresponde. También se
encontró que pipeline_completo() no le pasaba conservar_sin_cliente
a limpiar_datos() — corregido.

In [65]:
df_rfm, log_rfm = pipeline_completo('../data/raw/online_retail_II.xlsx', 
                                     '../data/processed/online_retail_rfm.parquet')
display(log_rfm)

                               paso   filas  facturas
                            inicial 1044848     53628
 sin compras con cancelación exacta 1043348     53404
                  sin cancelaciones 1024183     45112
              cantidad y precio > 0 1018154     39849
            sin códigos no-producto 1013721     39398
sin duplicados (cantidades sumadas)  991564     39398
              sin descripción vacía  991564     39398
                    con customer_id  764956     36490


,paso,filas,facturas
0,inicial,1044848,53628
1,sin compras con cancelación exacta,1043348,53404
2,sin cancelaciones,1024183,45112
3,cantidad y precio > 0,1018154,39849
4,sin códigos no-producto,1013721,39398
5,sin duplicados (cantidades sumadas),991564,39398
6,sin descripción vacía,991564,39398
7,con customer_id,764956,36490


**Hallazgo:** dataset para RFM: 766.521 filas, 36.650 facturas.
Se pierden 2.930 facturas al exigir customer_id (7,4% de las
39.580 que quedaban en ese punto del pipeline).

In [71]:
df_modelo, log_modelo = pipeline_completo('../data/raw/online_retail_II.xlsx',
                                            '../data/processed/online_retail_modelo.parquet',
                                            conservar_sin_cliente=True)
display(log_modelo)

                               paso   filas  facturas
                            inicial 1044848     53628
 sin compras con cancelación exacta 1043348     53404
                  sin cancelaciones 1024183     45112
              cantidad y precio > 0 1018154     39849
            sin códigos no-producto 1013721     39398
sin duplicados (cantidades sumadas)  991564     39398
              sin descripción vacía  991564     39398


,paso,filas,facturas
0,inicial,1044848,53628
1,sin compras con cancelación exacta,1043348,53404
2,sin cancelaciones,1024183,45112
3,cantidad y precio > 0,1018154,39849
4,sin códigos no-producto,1013721,39398
5,sin duplicados (cantidades sumadas),991564,39398
6,sin descripción vacía,991564,39398


**Hallazgo:** el dataset del modelo (conservar_sin_cliente=True)
tiene 993.249 filas y 39.580 facturas — 2.930 facturas más que el
de RFM (766.521 filas, 36.650 facturas), confirmando que se
recuperan las ventas sin cliente identificado para el modelo de
recomendación, sin afectar el dataset usado para RFM.

### 13. Revisión del PR
La revisión del PR de calidad-datos encontró varios puntos
reales. Ya se corrigieron en clean.py: solapamiento entre hojas del
Excel (1.088 facturas de diciembre 2010 duplicadas), el bloque
`__main__` que generaba un solo archivo en vez de los 2
documentados, y faltaba `openpyxl` en requirements.txt. Lo que
sigue acá es lo que necesitaba verificación con datos antes de
decidir el fix.

### 13.1 Códigos raros que sobrevivieron a la limpieza

In [72]:
codigos_raros_sobrevivientes = df_modelo[~df_modelo['stock_code'].str.match(r'^\d{5}$')]['stock_code'].unique()  # mismo chequeo de la 7.1, ahora sobre el dataset YA limpio
print(len(codigos_raros_sobrevivientes))
print(sorted(codigos_raros_sobrevivientes))

1343
['10002R', '10123C', '10123G', '10124A', '10124G', '15044A', '15044B', '15044C', '15044D', '15056BL', '15056N', '15056P', '15058A', '15058B', '15058C', '15059A', '15060B', '16020C', '16151A', '16156L', '16156N', '16156S', '16161C', '16161E', '16161G', '16161M', '16161P', '16161U', '16162L', '16162M', '16168M', '16168S', '16169A', '16169C', '16169D', '16169E', '16169F', '16169G', '16169K', '16169M', '16169N', '16169P', '16201A', '16201B', '16202A', '16202B', '16202C', '16202E', '16206B', '16207A', '16207B', '16243B', '16244A', '16244B', '16245A', '16248B', '16256B', '16256C', '16258A', '16258B', '17007B', '17011A', '17011B', '17011C', '17011D', '17011E', '17011F', '17012A', '17012B', '17012C', '17012D', '17012E', '17012F', '17013B', '17013C', '17013D', '17013F', '17014A', '17014C', '17014D', '17014E', '17014F', '17027A', '17028J', '17084A', '17084G', '17084J', '17084N', '17084P', '17084R', '17090A', '17090D', '17091A', '17091B', '17091F', '17091J', '17107D', '17108D', '17109A', '17

**Hallazgo:** 1.343 códigos no son 5 dígitos — la gran mayoría son
variantes reales de color/tamaño, mismo patrón que ya veníamos
confirmando (ej. 79323B, 79323G, 79323GR, 79323LP). Dos saltan
como candidatos reales a código administrativo: **TEST001** y
**TEST002** — el nombre sugiere entradas de prueba del sistema.
De paso, se corrige algo mal asumido antes: los códigos DCGS*
(DCGS0066N, DCGS0068, etc.) sí tienen descripciones de producto
real ("CAMOUFLAGE DOG COLLAR") — DCGS no es un prefijo
administrativo en general, la sospecha inicial era de una muestra
chica.

### 13.2 Verificando los candidatos a código administrativo

In [73]:
candidatos = ['TEST001', 'TEST002', 'DCGSSBOY', 'DCGSSGIRL']
display(df_modelo[df_modelo['stock_code'].isin(candidatos)][['stock_code', 'description', 'unit_price']].drop_duplicates())  # miro sus descripciones y precios reales

,stock_code,description,unit_price
26464,TEST001,This is a test product.,4.50
37434,TEST002,This is a test product.,1.00
230798,DCGSSBOY,BOYS PARTY BAG,3.36
230799,DCGSSGIRL,GIRLS PARTY BAG,3.36
554708,DCGSSBOY,BOYS PARTY BAG,3.29
554709,DCGSSGIRL,GIRLS PARTY BAG,3.29
740488,DCGSSBOY,BOYS PARTY BAG,1.25
740489,DCGSSGIRL,GIRLS PARTY BAG,1.25


**Hallazgo:** TEST001 y TEST002 tienen descripción literal "This is
a test product." — confirmado que son entradas de prueba del
sistema. DCGSSBOY y DCGSSGIRL son productos reales ("BOYS/GIRLS
PARTY BAG"), con 3 precios distintos a lo largo del tiempo —
patrón normal de un producto real, no administrativo.

**Decisión:** se agregan TEST001 y TEST002 a CODIGOS_NO_PRODUCTO.
DCGSSBOY y DCGSSGIRL no se tocan.

### 13.3 La compra de 80.995 unidades, ¿tiene su cancelación exacta todavía adentro?

In [74]:
cancelaciones_grandes = df[(df['quantity'] == -80995)]  # busco la cancelación de esa cantidad exacta
compras_grandes = df[(df['quantity'] == 80995) & (~df['invoice_no'].astype(str).str.startswith('C'))]  # y la compra original (no cancelación) de la misma cantidad
display(cancelaciones_grandes)
display(compras_grandes)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom


**Hallazgo:** confirmado el caso sospechado en la revisión del PR —
la compra de 80.995 unidades (factura 581483, 09:15) tiene su
cancelación exacta 12 minutos después (factura C581484, 09:27),
mismo producto (23843) y mismo cliente (16446). Hoy el filtro solo
saca la cancelación, dejando la compra original en el dataset
limpio como si fuera una venta real que nunca se concretó.

**Decisión:** agregar a limpiar_datos() un paso que saque también
la compra original cuando existe una cancelación con mismo
customer_id, stock_code y cantidad exacta (en valor absoluto) —
pendiente de medir el alcance real antes de escribir el fix.

### 13.4 ¿Cuántos casos como este hay en todo el dataset?

In [75]:
cancelaciones = df[df['invoice_no'].astype(str).str.startswith('C')].copy()  # todas las filas de cancelación
cancelaciones['quantity_abs'] = cancelaciones['quantity'].abs()  # paso la cantidad a positivo para poder comparar

compras = df[~df['invoice_no'].astype(str).str.startswith('C')].copy()  # todas las filas que NO son cancelación

match = compras.merge(
    cancelaciones[['customer_id', 'stock_code', 'quantity_abs']],
    left_on=['customer_id', 'stock_code', 'quantity'],
    right_on=['customer_id', 'stock_code', 'quantity_abs']
)  # cruzo compras con cancelaciones que coincidan en cliente, producto y cantidad
print(len(match), "compras con cancelación exacta (mismo cliente, producto y cantidad)")
print(match['quantity'].describe())

53331 compras con cancelación exacta (mismo cliente, producto y cantidad)
count    53331.000000
mean        12.128106
std        487.907513
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max      80995.000000
Name: quantity, dtype: float64


**Hallazgo:** el cruce simple (mismo cliente, producto y cantidad,
sin mirar fecha) encuentra 53.331 coincidencias — pero la mediana de
cantidad es 1, sugiriendo que la mayoría son compras repetidas
normales, no la misma operación cancelada. Hay que verificar con la
fecha antes de confiar en este número.

### 13.5 Verificando con la fecha: ¿la cancelación es siempre después de la compra?

In [76]:
match_con_fechas = compras.merge(
    cancelaciones[['customer_id', 'stock_code', 'quantity_abs', 'invoice_date']],
    left_on=['customer_id', 'stock_code', 'quantity'],
    right_on=['customer_id', 'stock_code', 'quantity_abs'],
    suffixes=('_compra', '_cancelacion')
)  # mismo cruce de antes, pero esta vez trayendo también la fecha de cada lado
match_con_fechas['horas_hasta_cancelar'] = (
    match_con_fechas['invoice_date_cancelacion'] - match_con_fechas['invoice_date_compra']
).dt.total_seconds() / 3600  # diferencia en horas entre la compra y la "cancelación"

print(match_con_fechas['horas_hasta_cancelar'].describe())

count    53331.000000
mean       613.691226
std       5863.045038
min     -17568.666667
25%      -2688.816667
50%        225.883333
75%       4173.116667
max      17521.316667
Name: horas_hasta_cancelar, dtype: float64


**Hallazgo:** el mínimo es -17.568 horas (~732 días) y el percentil
25 completo es negativo — hay cancelaciones que aparecen ANTES que
la compra que supuestamente cancelan. Esto confirma que el cruce
sin fecha empareja coincidencias sin relación real, no la misma
operación. La mediana general es 225,9 horas (~9,5 días) — muy
lejos del caso confirmado (80.995 unidades, 12 minutos).

### 13.6 Filtrando por tiempo positivo — ¿hay un patrón real de cancelación rápida escondido en el ruido?

In [77]:
match_positivos = match_con_fechas[match_con_fechas['horas_hasta_cancelar'] > 0]  # me quedo solo con los casos donde la cancelación es DESPUÉS de la compra
print(len(match_positivos), "de", len(match_con_fechas), "tienen la cancelación después de la compra")
print(match_positivos['horas_hasta_cancelar'].describe())
print(match_positivos['horas_hasta_cancelar'].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

31154 de 53331 tienen la cancelación después de la compra
count    31154.000000
mean      4299.574559
std       3988.503269
min          0.016667
25%        790.487500
50%       3234.900000
75%       7032.445833
max      17521.316667
Name: horas_hasta_cancelar, dtype: float64
0.10      115.291667
0.25      790.487500
0.50     3234.900000
0.75     7032.445833
0.90    10032.430000
Name: horas_hasta_cancelar, dtype: float64


**Hallazgo:** incluso filtrando solo los casos donde la cancelación
es posterior, la mediana sigue siendo 3.234,9 horas (~135 días) y
el percentil 10 es 115 horas (~5 días) — lejos todavía del caso
confirmado de 12 minutos. Hay que mirar el extremo más bajo de la
distribución.

### 13.7 ¿Hay un grupo real de cancelaciones rápidas en el extremo más bajo?

In [78]:
print(match_positivos['horas_hasta_cancelar'].quantile([0.01, 0.02, 0.05]))  # miro bien al principio de la distribución
print((match_positivos['horas_hasta_cancelar'] < 1).sum(), "casos con cancelación dentro de la primera hora")
print((match_positivos['horas_hasta_cancelar'] < 24).sum(), "casos con cancelación dentro de las primeras 24 horas")
print((match_positivos['horas_hasta_cancelar'] < 24*7).sum(), "casos con cancelación dentro de la primera semana")

0.01    0.066667
0.02    0.166667
0.05    1.983333
Name: horas_hasta_cancelar, dtype: float64
1356 casos con cancelación dentro de la primera hora
1926 casos con cancelación dentro de las primeras 24 horas
3824 casos con cancelación dentro de la primera semana


**Hallazgo:** 1.356 casos se cancelan dentro de la primera hora,
1.926 dentro de las primeras 24 horas, 3.824 dentro de la primera
semana — números que sugieren un grupo real concentrado en tiempos
cortos, a confirmar con la densidad por hora en cada tramo.

### 13.8 Afinando el corte — ¿dónde exactamente baja la densidad a ritmo de fondo?

In [79]:
bins = [0, 1/6, 0.5, 1, 6, 24, 48, 168]  # cortes en horas: 10min, 30min, 1h, 6h, 24h, 48h, 1 semana
labels = ['<10min', '10-30min', '30min-1h', '1h-6h', '6h-24h', '24h-48h', '48h-1sem']
duracion_horas = [1/6, 1/3, 0.5, 5, 18, 24, 120]  # duración real de cada tramo, para calcular la tasa por hora

match_positivos['bucket'] = pd.cut(match_positivos['horas_hasta_cancelar'], bins=bins, labels=labels)  # clasifico cada caso en su tramo
conteo = match_positivos['bucket'].value_counts().reindex(labels)  # cuántos casos caen en cada tramo
tasa_por_hora = conteo / duracion_horas  # normalizo por duración, para comparar tramos de distinto largo

print(conteo)
print()
print("Casos por hora en cada ventana:")
print(tasa_por_hora.round(1))

bucket
<10min       633
10-30min     572
30min-1h     180
1h-6h        332
6h-24h       209
24h-48h      266
48h-1sem    1634
Name: count, dtype: int64

Casos por hora en cada ventana:
bucket
<10min      3798.0
10-30min    1716.0
30min-1h     360.0
1h-6h         66.4
6h-24h        11.6
24h-48h       11.1
48h-1sem      13.6
Name: count, dtype: float64


**Hallazgo:** la tasa de coincidencias por hora cae en cascada desde
<10 minutos (3.798/hora) hasta 1-6 horas (66,4/hora), y ahí se
aplana en un nivel de fondo parejo (11-14/hora) que se mantiene
igual hasta la semana — ese nivel parejo es ruido de coincidencias
sin relación real, no cancelaciones genuinas.

**Decisión:** se usa una ventana de 6 horas para identificar una
compra como "cancelada": mismo customer_id, stock_code y cantidad
(en valor absoluto), con la cancelación ocurriendo dentro de las 6
horas posteriores a la compra. Fuera de esa ventana, la coincidencia
se trata como casual, no se excluye.

### 13.9 Probando el fix antes de regenerar los datasets completos

In [80]:
import importlib
from src.data import clean
importlib.reload(clean)  # tomo la versión más nueva de clean.py, con el fix ya aplicado

import pandas as pd
prueba = pd.DataFrame({
    'invoice_no': ['1', '2'], 'stock_code': ['A', 'B'], 'description': ['x', 'y'],
    'quantity': [1, 2], 'invoice_date': pd.to_datetime(['2020-01-01', '2020-01-01']),
    'unit_price': [1.0, 2.0], 'customer_id': [1, 2], 'country': ['UK', 'UK']
})  # caso mínimo, solo para confirmar que no hay errores de sintaxis o lógica
resultado, log = clean.limpiar_datos(prueba)
display(log)
print("Sin errores")

,paso,filas,facturas
0,inicial,2,2
1,sin compras con cancelación exacta,2,2
2,sin cancelaciones,2,2
3,cantidad y precio > 0,2,2
4,sin códigos no-producto,1,1
5,sin duplicados (cantidades sumadas),1,1
6,sin descripción vacía,1,1
7,con customer_id,1,1


Sin errores


**Hallazgo:** el código corrió sin errores sobre un caso mínimo. La
fila con stock_code 'B' se excluyó por el filtro de códigos
no-producto (coincidencia del dato de prueba con un código real ya
excluido), confirmando que ese filtro sigue funcionando como
corresponde.

### 13.10 Regenerando los datasets con todos los fixes de la revisión del PR

In [81]:
df_rfm, log_rfm = clean.pipeline_completo('../data/raw/online_retail_II.xlsx', 
                                            '../data/processed/online_retail_rfm.parquet')
display(log_rfm)

                               paso   filas  facturas
                            inicial 1044848     53628
 sin compras con cancelación exacta 1043348     53404
                  sin cancelaciones 1024183     45112
              cantidad y precio > 0 1018154     39849
            sin códigos no-producto 1013721     39398
sin duplicados (cantidades sumadas)  991564     39398
              sin descripción vacía  991564     39398
                    con customer_id  764956     36490


,paso,filas,facturas
0,inicial,1044848,53628
1,sin compras con cancelación exacta,1043348,53404
2,sin cancelaciones,1024183,45112
3,cantidad y precio > 0,1018154,39849
4,sin códigos no-producto,1013721,39398
5,sin duplicados (cantidades sumadas),991564,39398
6,sin descripción vacía,991564,39398
7,con customer_id,764956,36490


In [ ]:
df_modelo, log_modelo = clean.pipeline_completo('../data/raw/online_retail_II.xlsx',
                                                   '../data/processed/online_retail_modelo.parquet',
                                                   conservar_sin_cliente=True)
display(log_modelo)

                               paso   filas  facturas
                            inicial 1044848     53628
 sin compras con cancelación exacta 1043348     53404
                  sin cancelaciones 1024183     45112
              cantidad y precio > 0 1018154     39849
            sin códigos no-producto 1013721     39398
sin duplicados (cantidades sumadas)  991564     39398
              sin descripción vacía  991564     39398


,paso,filas,facturas
0,inicial,1044848,53628
1,sin compras con cancelación exacta,1043348,53404
2,sin cancelaciones,1024183,45112
3,cantidad y precio > 0,1018154,39849
4,sin códigos no-producto,1013721,39398
5,sin duplicados (cantidades sumadas),991564,39398
6,sin descripción vacía,991564,39398


**Hallazgo:** el paso nuevo "sin compras con cancelación exacta"
saca filas antes incluso de llegar a "sin cancelaciones" — y "sin
duplicados (cantidades sumadas)" ahora saca menos que antes del fix
del solapamiento de hojas (22.157 filas, contra 44.178 antes): buena
parte de lo que parecían duplicados reales dentro de una factura
eran en realidad copias entre las 2 hojas del Excel, sumadas dos
veces por error. Arreglar el solapamiento resolvió, de rebote, un
problema más grande de lo que parecía a simple vista.